# DLsite 音声作品自动翻译

DLsite 日语音声作品 → 中日双语 SRT 字幕

**支持多音频文件，共用背景信息，各自独立翻译。**

**运行前确认**：菜单栏 → 修改 → 笔记本设置 → 硬件加速器选 T4 GPU

依次运行每个格子即可。

In [ ]:
# ① 克隆仓库 & 安装依赖
# 请将下方 URL 替换为你自己的仓库地址
!git clone https://github.com/ushi-C/DLtrans.git
%cd DLtrans
!pip install -r requirements.txt -q
!nvidia-smi

In [ ]:
# ② 导入模块 & 初始化
import sys
sys.path.insert(0, '/content/DLsiteAudioTranslator')

from pathlib import Path
from google.colab import files
from api_client import init_openai_client, usage_stats
from asr import run_asr
from background import AudioBackground
from proofread import run_smart_proofread
from translate import run_parallel_translation
from utils import format_srt_time

client = init_openai_client()

In [ ]:
# ③ 输入背景信息（所有音频共用）
# 包括：作品标题、故事背景、登场角色
background = AudioBackground.input_interactive()

In [ ]:
# ========================================================
# 🚀 ④ 从谷歌网盘（Google Drive）秒速拉取音频文件（可多选）
# ========================================================
from google.colab import drive
import os

# 1. 自动挂载网盘（如果已连接则自动跳过）
if not os.path.exists('/content/drive'):
    print("正在连接你的谷歌网盘，请在页面弹出的窗口中点击“允许访问/连接”...")
    drive.mount('/content/drive')
else:
    print("✅ 谷歌网盘已处于连接状态")

# 2. 设置你在网盘中存放音频的文件夹名称（纯英文）
FOLDER_NAME = 'audio_in'
drive_folder_path = f'/content/drive/MyDrive/{FOLDER_NAME}'

# 如果网盘里还没有这个文件夹，自动帮你建一个
if not os.path.exists(drive_folder_path):
    os.makedirs(drive_folder_path)
    print(f"\n💡 提示：已自动在你的谷歌网盘根目录下创建了【{FOLDER_NAME}】文件夹。")
    print(f"👉 请现在去你的谷歌网盘，把音频文件拖进【{FOLDER_NAME}】文件夹中，然后重新运行本单元格！")
    audio_files = []
else:
    # 3. 扫描文件夹中的符合格式的音频文件
    allowed_extensions = ('.mp3', '.m4a', '.wav', '.mp4')

    # 获取网盘中所有符合后缀的文件，并转换成 Colab 能识别的完整路径
    audio_files = [
        os.path.join(drive_folder_path, f)
        for f in os.listdir(drive_folder_path)
        if f.lower().endswith(allowed_extensions)
    ]

    # 4. 打印结果，无缝对接后面的翻译步骤
    print('\n' + '='*50)
    if len(audio_files) > 0:
        print(f'✅ 成功！共从网盘拉取到 {len(audio_files)} 个音频文件：')
        for i, file_path in enumerate(audio_files, 1):
            print(f"   [{i}] {os.path.basename(file_path)}")
    else:
        print(f'⚠️ 提示：网盘中的【{FOLDER_NAME}】文件夹目前是空的。')
        print(f"👉 请先把音频文件上传到网盘的【{FOLDER_NAME}】文件夹里，再重新运行这里。")
    print('='*50)

In [ ]:
# ⑤ 粘贴トラックリスト → 正则拆轨 → 对齐到文件名 → 确认
# 交互：粘贴整份リスト后单独一行输入 END
# 也可非交互：把列表赋给 TRACKLIST 后用 apply_tracklist
#
# TRACKLIST = """
# 1. タイトル [01:17]
# タグ
# 2. タイトル [36:52]
# タグ
# """
# track_descs = AudioBackground.apply_tracklist(TRACKLIST, audio_files)

track_descs = AudioBackground.input_track_descriptions(audio_files)
background.track_descriptions = track_descs


In [ ]:
# ⑥ 逐个处理：ASR → 校对 → 翻译 → 写 SRT

srt_files = []

for audio_path in audio_files:
    filename = Path(audio_path).name
    print(f"\n{'='*50}")
    print(f"🎬 处理: {filename}")
    print(f"{'='*50}")

    # Step 1: ASR
    raw_asr = run_asr(audio_path)
    if not raw_asr:
        print(f"⚠️ {filename} 未识别到任何内容，跳过")
        continue

    # Step 2: 校对（背景辅助）
    proofed_data = run_smart_proofread(client, raw_asr, background, filename)

    # Step 3: 翻译
    final_data = run_parallel_translation(client, proofed_data, background, filename)

    # Step 4: 写 SRT
    srt_file = f"{Path(audio_path).stem}_bilingual.srt"
    with open(srt_file, 'w', encoding='utf-8') as f:
        for i, s in enumerate(final_data, 1):
            f.write(
                f"{i}\n"
                f"{format_srt_time(s['start'])} --> {format_srt_time(s['end'])}\n"
                f"{s['ja']}\n{s['zh']}\n\n"
            )

    print(f"📄 已生成: {srt_file}")
    srt_files.append(srt_file)

# 汇总
print(f"\n{'='*50}")
print(f"✅ 全部完成！共生成 {len(srt_files)} 个 SRT 文件")
print(f"   Token 消耗估算: {usage_stats.total_tokens}")
print(f"{'='*50}")

# 逐个下载
for srt_file in srt_files:
    files.download(srt_file)